# QA — CI/CD with GitHub Actions

**Duration:** 5 min

## GitHub Actions Pipeline

In [ ]:
name: CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }
      - run: pip install -r requirements.txt pytest
      - run: pytest tests/ -v

  build-and-push:
    needs: test
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4
      - name: Build Docker image
        run: docker build -t churn-api:${{ github.sha }} .
      - name: Push to ECR
        env:
          AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
          AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
        run: |
          aws ecr get-login-password --region us-east-1 | \
            docker login --username AWS --password-stdin ${{ secrets.ECR_REGISTRY }}
          docker tag churn-api:${{ github.sha }} ${{ secrets.ECR_REGISTRY }}/churn-api:latest
          docker push ${{ secrets.ECR_REGISTRY }}/churn-api:latest

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/mlops-real-world/mlops-qa-cicd.ipynb)


> **💡 Tip:** The pipeline: push to main → run tests → if tests pass → build Docker image → push to AWS ECR. No manual steps.